# Course 4: Large Language Models
## Session 4: Efficient training and inference

**Goal:** quantize a small model to 8, 4 and 2 bits, measure what is lost, and time generation with and without a KV cache.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) with tiny models and a small synthetic Bangla corpus. Cells labelled NOT RUN need a GPU, a model download or an account; run them on Kaggle or Colab.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class Config:
    vocab_size: int = 8000
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 4
    d_model: int = 256
    dropout: float = 0.0

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n_head = c.n_head
        self.ln1 = nn.LayerNorm(c.d_model)
        self.qkv = nn.Linear(c.d_model, 3 * c.d_model)
        self.proj = nn.Linear(c.d_model, c.d_model)
        self.ln2 = nn.LayerNorm(c.d_model)
        self.fc = nn.Linear(c.d_model, 4 * c.d_model)
        self.out = nn.Linear(4 * c.d_model, c.d_model)
        self.drop = nn.Dropout(c.dropout)

    def attn(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        split = lambda t: t.view(B, T, self.n_head, D // self.n_head).transpose(1, 2)
        y = F.scaled_dot_product_attention(split(q), split(k), split(v), is_causal=True)
        return self.proj(y.transpose(1, 2).reshape(B, T, D))

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.out(F.gelu(self.fc(self.ln2(x)))))
        return x

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.tok = nn.Embedding(c.vocab_size, c.d_model)
        self.pos = nn.Embedding(c.block_size, c.d_model)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln = nn.LayerNorm(c.d_model)
        self.head = nn.Linear(c.d_model, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.apply(self._init)

    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None, return_hidden=False):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for block in self.blocks:
            x = block(x)
        h = self.ln(x)
        if return_hidden:
            return h
        logits = self.head(h)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.9):
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.c.block_size:])
            probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
            sp, si = torch.sort(probs, descending=True)
            keep = torch.cumsum(sp, dim=-1) - sp < top_p
            sp = sp * keep
            sp = sp / sp.sum(dim=-1, keepdim=True)
            nxt = si.gather(-1, torch.multinomial(sp, 1))
            idx = torch.cat([idx, nxt], dim=1)
        return idx

In [ ]:
import random, itertools
SUBJ = ["মা", "বাবা", "শিক্ষক", "ছেলেটি", "মেয়েটি", "কৃষক", "দাদু", "রহিম"]
TIME = ["সকালে", "বিকেলে", "রাতে", "প্রতিদিন", "আজ"]
PLACE = ["বাজারে", "স্কুলে", "মাঠে", "বাড়িতে", "গ্রামে"]
OBJ_VERB = [("ভাত", "খায়"), ("বই", "পড়ে"), ("গান", "গায়"), ("ফুটবল", "খেলে"), ("চিঠি", "লেখে"), ("মাছ", "কেনে"), ("গল্প", "বলে"), ("ছবি", "আঁকে")]
VERB_OF = dict(OBJ_VERB)
# A toy language: subject, time, place, object, verb, full stop. The verb must fit the object (ভাত খায়, বই পড়ে).
ALL_SENTENCES = [[s, t, p, o, v, "।"] for s, t, p, (o, v) in itertools.product(SUBJ, TIME, PLACE, OBJ_VERB)]

def is_valid(words):
    return (len(words) == 6 and words[0] in SUBJ and words[1] in TIME and words[2] in PLACE
            and VERB_OF.get(words[3]) == words[4] and words[5] == "।")

EXTRA = ["<pad>", "<mask>", "প্রশ্ন", "উত্তর", ":", "?", "কী", "করে"]
VOCAB = EXTRA + SUBJ + TIME + PLACE + [o for o, _ in OBJ_VERB] + [v for _, v in OBJ_VERB] + ["।"]
STOI = {w: i for i, w in enumerate(VOCAB)}
enc = lambda words: [STOI[w] for w in words]
dec = lambda ids: [VOCAB[i] for i in ids]

In [ ]:
random.seed(0); torch.manual_seed(0)
_shuffled = ALL_SENTENCES[:]; random.shuffle(_shuffled)
train_sents, val_sents = _shuffled[:1200], _shuffled[1200:]            # 400 sentences the model never sees
train_ids = torch.tensor([i for s in train_sents for i in enc(s)])
val_ids = torch.tensor([i for s in val_sents for i in enc(s)])

def get_batch(data, block_size, batch_size):
    ix = torch.randint(0, (len(data) - block_size - 1) // 6, (batch_size,)) * 6      # start on a sentence boundary
    x = torch.stack([data[i:i + block_size] for i in ix]); y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x, y

def train_lm(cfg, steps=600, lr=3e-3, data=None, seed=0):
    torch.manual_seed(seed); data = train_ids if data is None else data
    model = GPT(cfg); opt = torch.optim.AdamW(model.parameters(), lr=lr)
    for _ in range(steps):
        x, y = get_batch(data, cfg.block_size, 32)
        _, loss = model(x, y); opt.zero_grad(); loss.backward(); opt.step()
    return model

@torch.no_grad()
def val_loss(model, block_size=24):
    n = (len(val_ids) - 1) // block_size * block_size
    x = val_ids[:n].view(-1, block_size); y = val_ids[1:n + 1].view(-1, block_size)
    return model(x, y)[1].item()

@torch.no_grad()
def sample_sentences(model, n=200, temperature=1.0):
    prompt = torch.tensor([enc(train_sents[i]) for i in torch.randint(len(train_sents), (n,)).tolist()])   # one real sentence as the prompt
    out = model.generate(prompt, 6, temperature=temperature, top_p=1.0)[:, 6:]                     # the model writes the next one
    return [dec(row.tolist()) for row in out]

In [ ]:
cfg = Config(vocab_size=len(VOCAB), block_size=24, n_layer=2, n_head=2, d_model=64)
base = train_lm(cfg)
print("base model trained | validation loss %.3f | valid sentences in 200 samples: %.2f" % (val_loss(base), sum(map(is_valid, sample_sentences(base))) / 200))

### 1. Quantize the weights
Round every weight to one of 2^b levels. Each row of a matrix gets its own scale, so one large weight does not ruin the whole matrix.

In [ ]:
import copy, numpy as np
def quantize(w, bits, per_row=True):
    levels = 2 ** (bits - 1) - 1
    scale = (w.abs().amax(dim=-1, keepdim=True) if per_row and w.dim() == 2 else w.abs().max()) / levels
    return torch.round(w / scale).clamp(-levels, levels) * scale

def quantized_copy(model, bits, per_row=True):
    m = copy.deepcopy(model)
    with torch.no_grad():
        for name, p in m.named_parameters():
            if p.dim() == 2: p.copy_(quantize(p, bits, per_row))          # matrices only; biases and norms are tiny
    return m

n_matrix = sum(p.numel() for p in base.parameters() if p.dim() == 2)
rows = []
print("bits   validation loss   valid samples   size of the matrices")
for bits in [32, 8, 6, 4, 3, 2]:
    m = base if bits == 32 else quantized_copy(base, bits)
    v = sum(map(is_valid, sample_sentences(m))) / 200
    rows.append((bits, val_loss(m), v)); print(f"{bits:4d}   {rows[-1][1]:15.3f}   {v:13.2f}   {n_matrix * bits / 8 / 1024:8.1f} KB")
assert abs(rows[1][1] - rows[0][1]) < 0.02 and rows[-1][1] > rows[0][1]

Eight bits costs almost nothing. Somewhere below 4 bits the model starts to break. Where exactly depends on the model: large models tolerate 4 bits far better than this tiny one, and methods such as GPTQ choose the rounding to protect the outputs instead of rounding each weight alone.

### 2. Per-row scales against one scale for the whole matrix

In [ ]:
for bits in [4, 3]:
    a, b = val_loss(quantized_copy(base, bits, per_row=True)), val_loss(quantized_copy(base, bits, per_row=False))
    print(f"{bits} bits: per-row scale {a:.3f} | one scale per matrix {b:.3f}")

### 3. The KV cache
Without a cache, generating token t recomputes attention for all t earlier tokens. With a cache, the keys and values of earlier tokens are stored and only the new token is processed. The results are identical; only the time differs.

In [ ]:
import time
@torch.no_grad()
def generate_cached(model, idx, n_new):
    """Greedy generation that feeds one token at a time and keeps each block's keys and values."""
    caches = [None] * len(model.blocks); out = idx.clone()
    def step(tok_id, pos):
        x = model.tok(tok_id) + model.pos(torch.tensor([pos]))
        for i, blk in enumerate(model.blocks):
            h = blk.ln1(x); B, T, D = h.shape
            q, k, v = blk.qkv(h).chunk(3, dim=-1)
            split = lambda t: t.view(B, -1, blk.n_head, D // blk.n_head).transpose(1, 2)
            q, k, v = split(q), split(k), split(v)
            if caches[i] is not None: k, v = torch.cat([caches[i][0], k], 2), torch.cat([caches[i][1], v], 2)
            caches[i] = (k, v)
            y = F.scaled_dot_product_attention(q, k, v)                   # the new token may look at everything stored
            x = x + blk.proj(y.transpose(1, 2).reshape(B, T, D)); x = x + blk.out(F.gelu(blk.fc(blk.ln2(x))))
        return model.head(model.ln(x))[:, -1]
    for pos in range(idx.size(1)): logits = step(idx[:, pos:pos + 1], pos)
    for pos in range(idx.size(1), idx.size(1) + n_new):
        nxt = logits.argmax(-1, keepdim=True); out = torch.cat([out, nxt], 1); logits = step(nxt, pos)
    return out

@torch.no_grad()
def generate_plain(model, idx, n_new):
    for _ in range(n_new): idx = torch.cat([idx, model(idx)[0][:, -1].argmax(-1, keepdim=True)], 1)
    return idx

big = GPT(Config(vocab_size=len(VOCAB), block_size=256, n_layer=4, n_head=4, d_model=128))      # untrained; only the timing matters
start = torch.tensor([enc(train_sents[0])])
t0 = time.perf_counter(); a = generate_plain(big, start, 200); t_plain = time.perf_counter() - t0
t0 = time.perf_counter(); b = generate_cached(big, start, 200); t_cache = time.perf_counter() - t0
print(f"200 tokens: no cache {t_plain:.2f} s | KV cache {t_cache:.2f} s | same output: {torch.equal(a, b)}")
assert torch.equal(a, b)

### 4. Speculative decoding in outline
A small draft model proposes a few tokens. The large model checks them all in one pass and keeps the longest prefix it agrees with. When the draft is usually right, most tokens cost one cheap pass and a share of one expensive pass.

In [ ]:
@torch.no_grad()
def speculative_greedy(target, draft, idx, n_new, k=4):
    passes = 0
    while n_new > 0:
        prop = generate_plain(draft, idx, min(k, n_new))                    # draft proposes k tokens
        check = target(prop)[0].argmax(-1)                                  # one pass of the large model scores all of them
        passes += 1; n_ok = 0
        for j in range(idx.size(1), prop.size(1)):
            if check[0, j - 1] == prop[0, j]: n_ok += 1
            else: break
        keep = prop[:, :idx.size(1) + n_ok]
        if n_ok < prop.size(1) - idx.size(1): keep = torch.cat([keep, check[:, keep.size(1) - 1:keep.size(1)]], 1)   # the large model's own token
        n_new -= keep.size(1) - idx.size(1); idx = keep
    return idx, passes

draft = train_lm(Config(vocab_size=len(VOCAB), block_size=24, n_layer=1, n_head=2, d_model=16), steps=300)
out, passes = speculative_greedy(base, draft, start, 18)
ref = generate_plain(base, start, 18); n_ref = ref.size(1)
print(f"large-model passes: {passes} instead of 18 | identical to greedy decoding with the large model alone: {torch.equal(out[:, :n_ref], ref)}")
assert torch.equal(out[:, :n_ref], ref)

### 5. Real 4-bit models (NOT RUN here)

In [ ]:
# NOT RUN: needs a GPU and `pip install transformers bitsandbytes accelerate`.
import time, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
name = "<a small open model, 1 to 3 billion parameters>"
tok = AutoTokenizer.from_pretrained(name)
full = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, device_map="auto")
q4 = AutoModelForCausalLM.from_pretrained(name, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
# For each model report: memory (model.get_memory_footprint()), tokens per second for 200 new tokens,
# and perplexity on 100 held-out Bangla sentences. Expect a large memory saving. Do not expect 4-bit to be faster on a GPU:
# bitsandbytes saves memory, while GGUF files run with llama.cpp are the usual route to fast CPU inference.

### TODO
1. Quantize only the embedding, or only the attention matrices, to 3 bits. Which part is most sensitive?
2. Muon in one step: for a random 64x64 gradient matrix, compute its orthogonal factor with the Newton-Schulz iteration `X <- 1.5 X - 0.5 X Xᵀ X` (after dividing by the largest singular value) and check that the singular values approach 1.
3. Run the 4-bit cell and fill in a table of memory, speed and perplexity.